# Detekcja krawędzi

## Cel ćwiczenia

- Zapoznanie z metodami detekcji krawędzi:
    - Sobel, Prewitt, Roberts - przypomnienie,
    - Laplasjan z Gaussa (LoG – ang. Laplacian of Gaussian),
    - Canny.

Detekcja krawędzi przez wiele lat była podstawą algorytmów segmentacji.
Krawędzie wykrywane są najczęściej z wykorzystaniem pierwszej (gradient) i drugiej (Laplasjan) pochodnej przestrzennej.
Wykorzystanie obu metod zaprezentowane zostało w ćwiczeniu *Przetwarzanie wstępne. Filtracja kontekstowa*.

W niniejszym ćwiczeniu poznane detektory krawędzi zostaną porównane z bardziej zaawansowanymi: Laplasjan z funkcji Gaussa (LoG), Zero Crossing i Canny.

## Laplasjan z Gaussa (LoG)

Funkcja Gaussa:<br>
\begin{equation}
h(r) = e^{\frac{-r^2}{2 \sigma^2}}
\end{equation}<br>
gdzie:
- $r^2 = x^2 + y^2$
- $\sigma$ to odchylenie standardowe.

Działanie filtracji Gaussowskiej zostało przedstawione w ćwiczeniu "Przetwarzanie wstępne". W jej wyniku następuje rozmazanie obrazu.
Laplasjan tej funkcji dany jest wzorem:

\begin{equation}
\nabla^2 h(r) = \frac{r^2 - 2\sigma^2}{\sigma^4} e^{-\frac{r^2}{2\sigma^2}}
\end{equation}

Funkcję (z oczywistych powodów) nazywamy Laplasjan z Gaussa (LoG).
Ponieważ druga pochodna jest operacją liniową, konwolucja obrazu z $\nabla^2 h(r)$ daje taki sam efekt jak zastosowanie filtracji Gaussa na obrazie, a następnie obliczenie Laplasjanu z wyniku.
Lokalizacja krawędzi polega na znalezieniu miejsca, gdzie po filtracji LoG następuje zmiana znaku.

1. Wczytaj obraz *house.png*.
2. Wykonaj rozmycie Gaussowskie obrazu wejściowego.
W tym celu wykorzystaj funkcję `cv2.GaussianBlur(img, kSize, sigma)`.
Pierwszy argument jest obrazem wejściowym.
Drugi jest rozmiarem filtru (podanym w nawiasach okrągłych, np. *(3, 3)*).
Trzecim argumentem jest odchylenie standardowe - zostanie dobrane automatycznie, jeśli argument ma wartość `0` (będzie równe rozmiarowi).
3. Oblicz laplasjan obrazu rozmytego.
W tym celu wykorzystaj funkcję `cv2.Laplacian(img, ddepth)`.
Pierszym argumentem jest obraz wejściowy.
Drugim argumentem jest typ danych wejściowych. Użyj `cv2.CV_32F`.
4. Wyznacz miejsca zmiany znaku.
Zaimplementuj funkcję `crossing(LoG, thr)`:
    - Najpierw stwórz tablicę, do której zostanie zapisany wynik.
    Jej rozmiar jest taki sam jak przetwarzanego obrazu.
    - Następnie wykonaj pętle po obrazie (bez ramki jednopikselowej).
    W każdej iteracji stwórz otoczenie o rozmiarze $3 \times 3$.
    Dla otoczenia oblicz wartość maksymalną i minimalną.
    - Jeśli wartości te mają przeciwne znaki, to do danego miejsca tablicy przypisz wartość:
        - jeśli piksel wejściowy > 0, to dodaj do niego wartość bezwzględną minimum.
        - jeśli piksel wejściowy < 0, to do jego wartości bezwzględnej dodaj maksimum.
    - Zmień zakres wykonanej tablicy do $<0, 255>$.
    - Wykonaj progowanie tablicy. Próg jest argumentem wejściowym.
    - Przeskaluj dane binarne do wartości `[0, 255]`.
    - Wykonaj konwersję do typu *uint8*.
    - Wykonaj filtrację medianową wyniku.
    Wykorzystaj funkcję `cv2.medianBlur(img, kSize)`.
    Pierwszym argumentem jest obraz wejśćiowy, a drugim rozmiar filtra.
    - Zwróć wyznaczoną tablicę.
5. Wyświetl obraz wynikowy.
6. Dobierz parametry (rozmiar filtru Gaussa, odchylenie standardowe, próg binaryzacji) tak, by widoczne były kontury domu, ale nie dachówki.

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import math
import os

if not os.path.exists("dom.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/09_Canny/dom.png --no-check-certificate

def crossing(LoG,thr):
    h,w = LoG.shape
    out = np.zeros((h,w), dtype = np.float32)

    for y in range(1,h-1):
        for x in range(1,w-1):
            fragment = LoG[y-1:y+2, x - 1:x + 2]
            max_val = np.max(fragment)
            min_val = np.min(fragment)

            input_pixel = LoG[y,x]

            if max_val > 0 and min_val < 0:
                if input_pixel > 0:
                    out[y,x] = input_pixel + abs(min_val)
                elif input_pixel < 0:
                    out[y,x] = abs(input_pixel) + max_val
                else:
                    out[y,x] = 0
    
    out = cv2.normalize(out,None,0,255,cv2.NORM_MINMAX)
    # progowanie
    _, out = cv2.threshold(out, thr, 255, cv2.THRESH_BINARY)

    # uint8 + filtracja medianowa
    out = out.astype(np.uint8)
    out = cv2.medianBlur(out, 3)

    return out

img = cv2.imread("dom.png", cv2.IMREAD_GRAYSCALE)


blur = cv2.GaussianBlur(img, (7, 7), 1.4)

log_img = cv2.Laplacian(blur, cv2.CV_32F)


result = crossing(log_img, 70)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(img, cmap="gray")
plt.title("Oryginal")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(blur, cmap="gray")
plt.title("GaussianBlur")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(log_img, cmap="gray")
plt.title("Laplacian of Gaussian")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(result, cmap="gray")
plt.title("Wynik")
plt.axis("off")

plt.tight_layout()
plt.show()

## Algorytm Canny'ego

> Algorytm Canny'ego to często wykorzystywana metoda detekcji krawędzi.
> Zaproponowana została w~1986r - autorem był John F. Canny.
> Przy jego projektowaniu założono trzy cele:
> - niska liczba błędów - algorytm powinien znajdować wszystkie krawędzie oraz generować jak najmniej fałszywych detekcji,
> - punkty krawędziowe powinny być poprawnie lokalizowane - wykryte punkty powinny być jak najbardziej zbliżone do rzeczywistych,
> - krawędzie o szerokości 1 piksela - algorytm powinien zwrócić jeden punkt dla każdej rzeczywistej krawędzi.

Zaimplementuj pierwszą część algorytmu detekcji krawędzi Canny'ego:
1. W pierwszym kroku obraz przefiltruj dwuwymiarowym filtrem Gaussa.
2. Następnie oblicz gradient pionowy i poziomy ($g_x $ i $g_y$).
Jedną ze stosowanych metod jest gradient Sobela.
3. Dalej oblicz amplitudę:
$M(x,y)  = \sqrt{g_x^2+g_y^2}$ oraz kąt:
$\alpha(x,y) = arctan(\frac{g_y}{g_x})$.
Do obliczenia kąta wykorzystaj funkcję `np.arctan2(x1, x2)`.
Wynik jest w radianach.
4. W kolejnym etapie wykonaj kwantyzację kątów gradientu.
Kąty od $-180^\circ$ do $180^\circ$ można podzielić na 8 przedziałów:
[$-22.5^\circ, 22.5^\circ$], [$22.5^\circ, 67.5^\circ$],
[$67.5^\circ, 112.5^\circ$], [$112.5^\circ, 157.5^\circ$],
[$157.5^\circ, -157.5^\circ$], [$-157.5^\circ, -112.5^\circ$],
[$-112.5^\circ, -67.5^\circ$], [$-67.5^\circ, -22.5^\circ$].
Przy czym należy rozpatrywać tylko 4 kierunki:
    - pionowy ($d_1$),
    - poziomy ($d_2$),
    - skośny lewy ($d_3$),
    - skośny prawy ($d_4$).
5. Dalej przeprowadź eliminację pikseli, które nie mają wartości maksymalnej (ang. *nonmaximal suppresion*).
Celem tej operacji jest redukcja szerokości krawędzi do rozmiaru 1 piksela.
Algorytm przebiega następująco:
W rozpatrywanym otoczeniu o rozmiarze $3 \times 3$:
    - określ do którego przedziału należy kierunek gradientu piksela centralnego ($d_1, d_2, d_3, d_4$).
    - przeanalizuj sąsiadów leżących na tym kierunku.
Jeśli choć jeden z nich ma amplitudę większą niż piksel centralny, to należy uznać, że nie jest to lokalne maksimum i do wyniku przypisać $g_N(x,y) = 0$.
W przeciwnym przypadku $g_N(x,y) = M(x,y)$.
Przez $g_N$ rozumiemy obraz detekcji lokalnych maksimów.
Zaimplementuj funkcję `nonmax`.
Pierwszym argementem jest macierz kierunków (po kwantyzacji).
Drugim argumentem jest macierz amplitudy.
6. Ostatnią operacją jest binaryzacja obrazu $g_N$.
Stosuje się tutaj tzw. binaryzację z histerezą.
Wykorzystuje się w niej dwa progi: $T_L$ i $T_H$, przy czym $T_L < T_H$.
Canny zaproponował, aby stosunek progu wyższego do niższego był jak 3 lub 2 do 1.
Rezultaty binaryzacji można opisać jako:<br>
$g_{NH}(x,y) = g_N(x,y) \geq TH $<br>
$g_{NL}(x,y) = TH > g_N(x,y) \geq TL $<br>
Można powiedzieć, że na obrazie $g_{NH}$ są "pewne" krawędzie.
Natomiast na $g_{NL}$ "potencjalne".
7. Na jednym obrazie zaznacz piksele należące do obrazu $g_{NH}$ jako czerwone oraz należące do obrazu $g_{NL}$ jako niebieskie.
Wyświetl obraz.

In [ ]:
def quantize_angles(alpha_deg):
    h, w = alpha_deg.shape
    directions = np.zeros((h, w), dtype=np.uint8)

    for y in range(h):
        for x in range(w):
            a = alpha_deg[y, x]

            # d2 - poziomy (porownanie lewo/prawo)
            if (-22.5 <= a < 22.5) or (a >= 157.5) or (a < -157.5):
                directions[y, x] = 2

            # d4 - skos /
            elif (22.5 <= a < 67.5) or (-157.5 <= a < -112.5):
                directions[y, x] = 4

            # d1 - pionowy (porownanie gora/dol)
            elif (67.5 <= a < 112.5) or (-112.5 <= a < -67.5):
                directions[y, x] = 1

            # d3 - skos \
            else:
                directions[y, x] = 3

    return directions


def nonmax(directions, M):
    h, w = M.shape
    gN = np.zeros((h, w), dtype=np.float32)

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            d = directions[y, x]
            m = M[y, x]

            if d == 1:
                n1 = M[y - 1, x]
                n2 = M[y + 1, x]
            elif d == 2:
                n1 = M[y, x - 1]
                n2 = M[y, x + 1]
            elif d == 3:
                n1 = M[y - 1, x - 1]
                n2 = M[y + 1, x + 1]
            else:  # d == 4
                n1 = M[y - 1, x + 1]
                n2 = M[y + 1, x - 1]

            if m >= n1 and m >= n2:
                gN[y, x] = m
            else:
                gN[y, x] = 0

    return gN


def canny_stage1(img, gauss_ksize=(5, 5), gauss_sigma=1.0, TL=20, TH=40):
    if img is None:
        raise ValueError("Obraz wejsciowy jest pusty")
    if TL >= TH:
        raise ValueError("TL musi byc mniejsze od TH")

    blur = cv2.GaussianBlur(img, gauss_ksize, gauss_sigma)

    gx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)

    M = np.hypot(gx, gy)
    alpha = np.arctan2(gy, gx)
    alpha_deg = np.rad2deg(alpha)

    directions = quantize_angles(alpha_deg)

    gN = nonmax(directions, M)

    gN_norm = cv2.normalize(gN, None, 0, 255, cv2.NORM_MINMAX)
    gN_norm = gN_norm.astype(np.uint8)

    gNH = gN_norm >= TH
    gNL = (gN_norm >= TL) & (gN_norm < TH)

    h, w = img.shape
    vis = np.zeros((h, w, 3), dtype=np.uint8)  
    vis[gNL] = [0, 0, 255]      
    vis[gNH] = [255, 0, 0]      


    return {
        "blur": blur,
        "gx": gx,
        "gy": gy,
        "M": M,
        "alpha_deg": alpha_deg,
        "directions": directions,
        "gN": gN,
        "gN_norm": gN_norm,
        "gNH": gNH,
        "gNL": gNL,
        "vis": vis
    }


img = cv2.imread("dom.png", cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError("Nie udalo sie wczytac pliku dom.png")

result = canny_stage1(
    img,
    gauss_ksize=(7, 7),
    gauss_sigma=1.4,
    TL=30,
    TH = 90
)


plt.figure(figsize=(14, 8))

plt.subplot(2, 3, 1)
plt.imshow(img, cmap="gray")
plt.title("Oryginal")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(result["blur"], cmap="gray")
plt.title("Po Gaussie")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(result["M"], cmap="gray")
plt.title("Amplituda M")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(result["directions"], cmap="tab10")
plt.title("Kierunki po kwantyzacji")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(result["gN_norm"], cmap="gray")
plt.title("Po nonmax")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(result["vis"])
plt.title("Czerwone: gNH, Niebieskie: gNL")
plt.axis("off")

plt.tight_layout()
plt.show()



## Algorytm Canny'ego - OpenCV

1. Wykonaj detekcję krawędzi metodą Canny'ego wykorzystując funkcję `cv2.Canny`.
    - Pierwszym argumentem funkcji jest obraz wejściowy.
    - Drugim argumentem jest mniejszy próg.
    - Trzecim argumentem jest większy próg.
    - Czwarty argument to tablica, do której wpisany zostanie wynik.
    Można zwrócić go przez wartość i podać wartość `None`.
    - Piąty argument to rozmiar operatora Sobela (w naszym przypadku 3).
    - Szósty argument to rodzaj używanej normy.
    0 oznacza normę $L_1$, 1 oznacza normę $L_2$. Użyj $L_2$.
2. Wynik wyświetl i porównaj z wykonaną częściową implementacją w poprzednim ćwiczeniu.
Na czym polegają różnice?

In [ ]:
blur = cv2.GaussianBlur(img, (7, 7), 1.4)
cv2version = cv2.Canny(blur,30,90,None,3,1)
plt.figure(figsize=(6, 6))
plt.imshow(cv2version, cmap="gray")
plt.title("Canny Edge Detection (OpenCV)")
plt.axis("off")
plt.show()

plt.imshow(result["vis"])
plt.title("Czerwone: gNH, Niebieskie: gNL")
plt.axis("off")

moja implementacja realizuje wstępne etapy algorytmu Canny’ego: wygładzanie, gradient, kwantyzację kierunku, nonmax i podwójne progowanie,
funkcja cv2.Canny wykonuje dodatkowo pełną histerezę i zwraca końcową binarną mapę krawędzi,
dlatego wynik OpenCV jest bardziej zwarty i jednopikselowy, a mój pokazuje jeszcze rozróżnienie na krawędzie mocne i słabe.